# Evaluation — ranking and GraphRAG vs vector RAG

Two comparisons, each on a sample large enough to trust:

1. **Recommendation ranking** on the same per-user 80/20 split as `03_baseline_recommenders.ipynb`  
   Popularity vs item-CF vs `GraphRecommender` → Precision@K, Recall@K, HitRate@K, **nDCG@K**.  
   Write-up: `docs/graph_recommender.md`. This is the path ranker, **not** GraphRAG.
2. **Factual QA** on a shared gold set built from Neo4j  
   TF-IDF vector RAG vs GraphRAG → factual hit + whether the gold fact was in the retrieved / Cypher context.  
   A 7-question `llama3.2:3b` smoke test was inconclusive. With `qwen3:8b` on **30** questions, GraphRAG won on **accuracy** (29/30 vs 22/30) and **explainability** at the same retrieval coverage (~93%).  
   Write-up: `docs/graph_rag.md`.

Graph recommendations are **user-level**: take each user's top liked training titles as seeds, aggregate path scores, and drop movies already seen in train.

Requires MovieLens CSVs under `data/raw/…`. Ranking needs Neo4j only for the graph method. QA also needs Ollama.

**Run All is safe for the published snapshot.** `eval/latest_results.json` is only written by `scripts/run_evaluation.py`. Live cells below stay off (`RUN_* = False`) so this notebook does not re-query Neo4j / Ollama or replace the cached tables.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "eval"))

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:.3f}".format)

MAX_USERS = 200  # 0 = every user with a relevant held-out item
TOP_K = 10
N_SEEDS = 5  # liked train titles fed to GraphRecommender
# Live re-runs. Leave False so Run All keeps the published snapshot.
# Only scripts/run_evaluation.py writes eval/latest_results.json.
RUN_GRAPH_RANKER = False
RUN_QA = False

In [2]:
# Last CLI snapshot (no Neo4j / Ollama required)
import json

cached_path = ROOT / "eval" / "latest_results.json"
if cached_path.exists():
    cached = json.loads(cached_path.read_text(encoding="utf-8"))
    if cached.get("ranking"):
        print("Cached ranking")
        display(pd.DataFrame(cached["ranking"]))
    if cached.get("qa_summary"):
        print("Cached QA")
        display(pd.DataFrame(cached["qa_summary"]))
else:
    print("No eval/latest_results.json yet — run: uv run python scripts/run_evaluation.py --all")

Cached ranking


,method,n_users,precision@k,recall@k,hit_rate@k,ndcg@k,ndcg@k_se
0,popularity,200,0.012,0.010,0.105,0.012,0.003
1,item_based_cf,200,0.012,0.013,0.100,0.014,0.004
2,graph_recommender,200,0.048,0.049,0.340,0.071,0.009


Cached QA


,method,n,factual_accuracy,evidence_in_context,errors
0,graph_rag,30,0.967,0.933,0
1,vector_rag,30,0.733,0.933,0


## 1. Recommendation ranking

Same protocol as notebook 03:

- per-user holdout, `test_size=0.2`, `random_state=42`
- relevant = held-out rating ≥ 4.0
- K = 10
- plus **nDCG@K** (binary gain)

`graph_recommender` is slower because each user triggers a few Neo4j path queries. `RUN_GRAPH_RANKER` defaults to `False`; the published 200-user table is in the cached cell above and in `docs/graph_recommender.md`.

In [3]:
from recommend_eval import run_recommendation_eval

rec_summary, rec_detail, rec_paired = run_recommendation_eval(
    max_users=MAX_USERS,
    k=TOP_K,
    n_seeds=N_SEEDS,
    include_graph=RUN_GRAPH_RANKER,
)
print("Held-out ranking (higher is better)")
display(rec_summary)
print("Paired wins: graph vs each baseline on the same users")
display(pd.DataFrame(rec_paired))

Evaluating popularity on up to 200 users (K=10)...
  popularity: user 1/200
  popularity: user 25/200
  popularity: user 50/200
  popularity: user 75/200
  popularity: user 100/200
  popularity: user 125/200
  popularity: user 150/200
  popularity: user 175/200
  popularity: user 200/200
Evaluating item_based_cf on up to 200 users (K=10)...
  item_based_cf: user 1/200
  item_based_cf: user 25/200
  item_based_cf: user 50/200
  item_based_cf: user 75/200
  item_based_cf: user 100/200
  item_based_cf: user 125/200
  item_based_cf: user 150/200
  item_based_cf: user 175/200
  item_based_cf: user 200/200
Evaluating graph_recommender on up to 200 users (K=10, n_seeds=5)...
  graph_recommender: user 1/200
  graph_recommender: user 25/200
  graph_recommender: user 50/200
  graph_recommender: user 75/200
  graph_recommender: user 100/200
  graph_recommender: user 125/200
  graph_recommender: user 150/200
  graph_recommender: user 175/200
  graph_recommender: user 200/200
Held-out ranking (high

,method,n_users,precision@k,recall@k,hit_rate@k,ndcg@k,ndcg@k_se
0,popularity,200,0.012,0.010,0.105,0.012,0.003
1,item_based_cf,200,0.012,0.013,0.100,0.014,0.004
2,graph_recommender,200,0.048,0.049,0.340,0.071,0.009


Paired wins: graph vs each baseline on the same users


,left,right,metric,n_users,mean_delta,pct_left_better,pct_tie,pct_right_better
0,graph_recommender,popularity,ndcg_at_k,200,0.059,0.340,0.585,0.075
1,graph_recommender,popularity,hit_rate_at_k,200,0.235,0.310,0.615,0.075
2,graph_recommender,popularity,precision_at_k,200,0.036,0.320,0.605,0.075
3,graph_recommender,item_based_cf,ndcg_at_k,200,0.057,0.340,0.590,0.070
4,graph_recommender,item_based_cf,hit_rate_at_k,200,0.240,0.310,0.620,0.070
5,graph_recommender,item_based_cf,precision_at_k,200,0.036,0.315,0.615,0.070


### How to read the ranking table

- **Popularity** and **item-CF** are the notebook 03 baselines on this exact split.
- **graph_recommender** is *not* GraphRAG. It is the multi-signal path ranker (`DIRECTED_BY`, `ACTED_BY`, genres, keywords, co-fans).
- A higher nDCG@K means relevant held-out titles sit higher in the list, not only that they appear somewhere in the top K.
- On MovieLens Latest Small, popularity is a strong baseline because popular titles dominate held-out positives. The graph method is more useful when you also care about **why** a title was ranked.

Latest CLI run (`max_users=200`, `K=10`, `n_seeds=5`, same holdout as notebook 03):

| method | P@10 | R@10 | Hit@10 | nDCG@10 (SE) |
|---|---:|---:|---:|---:|
| popularity | 0.012 | 0.010 | 0.105 | 0.012 (0.003) |
| item-CF | 0.012 | 0.013 | 0.100 | 0.014 (0.004) |
| graph_recommender | **0.048** | **0.049** | **0.340** | **0.071 (0.009)** |

The n=20 / 3-seed run was too small and handicapped the graph (CF sees the full train history). With 200 users and 5 liked seeds the graph ranker clearly leads: about 4× Precision/Recall, Hit@10 34% vs ~10%, nDCG gap much larger than the standard error. On paired nDCG it beats each baseline for 34% of users and loses for ~7% (the rest are ties, usually both zero). This is still the path ranker, not GraphRAG. The GraphRAG vs vector RAG comparison is a different eval (`docs/graph_rag.md`).


## 2. QA — vector RAG vs GraphRAG

Gold questions are built from the live graph (**30** items: directors, genres, cast, shared director, shared actors), so the labels stay consistent with the loaded catalog. The first smoke test had only 7 questions on `llama3.2:3b` and was not enough. Full write-up: `docs/graph_rag.md`.

| Method | Retrieval | Answer |
|---|---|---|
| `vector_rag` | TF-IDF over title + overview + cast/crew/genre documents | Ollama, documents only |
| `graph_rag` | LLM writes Cypher, Neo4j returns rows | Ollama verbalizes those rows |

Metrics:

- **factual_correct** — gold name(s) appear in the answer
- **evidence_in_context** — the same name(s) appear in the retrieved documents or Cypher result

`RUN_QA` defaults to `False`. A live re-run needs Ollama + `qwen3:8b` and does **not** write `eval/latest_results.json`. The cell below may still show an older 7-question output; trust the cached table at the top and the markdown summary after it.

In [ ]:
if RUN_QA:
    from qa_eval import results_frame, run_qa_eval, summarize_qa

    gold, qa_results = run_qa_eval()
    qa_frame = results_frame(qa_results)
    print(f"Gold questions: {len(gold)}")
    display(summarize_qa(qa_frame))
    display(
        qa_frame[
            [
                "method",
                "id",
                "question",
                "expected",
                "factual_correct",
                "evidence_in_context",
                "answer",
                "error",
            ]
        ]
    )
else:
    print("QA skipped (set RUN_QA = True).")

Building gold questions from Neo4j...
QA model: llama3.2:3b
Loading movie documents for vector RAG (30 gold questions)...
Starting GraphRAG (schema refresh)...
QA director_interstellar: vector_rag...
QA director_interstellar: graph_rag...
QA director_inception: vector_rag...
QA director_inception: graph_rag...
QA director_matrix: vector_rag...
QA director_matrix: graph_rag...
QA director_pulp: vector_rag...
QA director_pulp: graph_rag...
QA director_shawshank: vector_rag...
QA director_shawshank: graph_rag...
QA director_godfather: vector_rag...
QA director_godfather: graph_rag...
QA director_dark_knight: vector_rag...
QA director_dark_knight: graph_rag...
QA director_spirited: vector_rag...
QA director_spirited: graph_rag...
QA genres_toy_story: vector_rag...
QA genres_toy_story: graph_rag...
QA genres_forrest: vector_rag...
QA genres_forrest: graph_rag...
QA genres_alien: vector_rag...
QA genres_alien: graph_rag...
QA genres_get_out: vector_rag...
QA genres_get_out: graph_rag...
QA g

### How to read the QA table

- GraphRAG should win on **relationship** questions (shared director / shared actors) if the generated Cypher is valid.
- Vector RAG can still answer “who directed X?” when the director string sits in a retrieved overview or metadata field.
- If `evidence_in_context` is true and `factual_correct` is false, retrieval found the fact but the LLM dropped it (the common vector-RAG failure on this set).
- If both are false on GraphRAG, inspect `retrieved_or_cypher` — the model likely wrote bad Cypher.
- Small local models often wrap Cypher in prose (`Here is the query…`). `graph_rag.strip_cypher_preamble` peels that off before Neo4j runs it. `llama3.2:3b` is not the published comparison.
- Latest CLI run (`qwen3:8b`, 30 gold questions from the live graph):

| method | n | factual | evidence in context |
|---|---:|---:|---:|
| graph_rag | 30 | **0.97** | 0.93 |
| vector_rag | 30 | 0.73 | 0.93 |

By type (factual accuracy):

| kind | graph_rag | vector_rag |
|---|---:|---:|
| director | 1.00 | 0.75 |
| genres | 1.00 | 0.67 |
| actors | 1.00 | 0.80 |
| shared_director | 1.00 | 1.00 |
| shared_actors | 0.83 | 0.50 |

GraphRAG's only factual miss was shared Godfather cast. Vector RAG often had the fact in the retrieved docs and still dropped it (Dark Knight director/cast, Forrest Gump genres, Toy Story / Batman / Ocean's shared actors). Retrieval coverage was similar (0.93). **GraphRAG won on factual accuracy and explainability**: the answer is grounded in explicit Cypher rows (who is connected to whom), not an opaque nearest-document snippet. Same coverage, better use of evidence, especially on multi-hop shared-cast questions.

CLI (this is what writes `eval/latest_results.json`):

```bash
uv run python scripts/run_evaluation.py --recs --max-users 200
uv run python scripts/run_evaluation.py --qa --model qwen3:8b
```

Full narrative: `docs/graph_rag.md`.
